# Appendix 06: Iterative Estimation Methods

Source orientation: printed pages 597-627; PDF pages 615-645.

This notebook is an original, standalone computational treatment of the appendix. The source pages were used only to identify the section hierarchy and mathematical targets: Newton and Gauss-Newton steps, Levenberg-Marquardt damping, sparse systems, robust costs, rotations, homogeneous vectors, and constrained parameterizations.

## Standalone Learning Goals

By the end of this appendix lab you should be able to:

- explain an iterative estimator as a sequence of local linear least-squares problems;
- compare Newton/Gauss-Newton style steps with damped Levenberg-Marquardt steps;
- verify an analytic Jacobian with finite differences;
- inspect why robust weights reduce the influence of outliers;
- see Schur-complement sparsity as the computational shape behind bundle adjustment.

## Library Routing

The appendix is about optimization structure rather than a single closed-form object. NumPy supplies the analytic Jacobians and small dense solves; SciPy sparse represents block normal equations; NetworkX makes the camera-point coupling visible; Matplotlib records optimizer paths, damping schedules, robust weights, and sparsity. This is intentionally not a generic plotting pass: each artifact corresponds to a named iterative-estimation concept and records a checkable invariant.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

artifact_paths = []


## Concept Map From Nonlinear Geometry To Local Linear Problems

Most MVG estimators minimize a nonlinear sum of residuals:

$$\min_\theta \sum_i \rho(r_i(\theta)^2).$$

An iteration replaces the nonlinear surface by a local model, solves a linearized problem, accepts or damps the step, and repeats. The important thing to inspect is not just the final parameter value. Watch the path, the objective, the Jacobian, the damping, and the graph of variable interactions.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from scipy import sparse

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib

TOPIC = "appendix-06"
rng = np.random.default_rng(106)
plt.rcParams.update({"figure.dpi": 120})


## 1. Optimizer paths: the same residual model can move cautiously or aggressively

The synthetic residual model fits a nonlinear exponential curve. Gauss-Newton follows the local least-squares approximation directly; Levenberg-Marquardt adds damping when that approximation is too bold. Inspect how the damped path takes smaller but more reliable turns across the objective contours.

In [ ]:
x = np.linspace(0.0, 2.2, 36)
true_theta = np.array([1.25, -0.85])
y = true_theta[0] * np.exp(true_theta[1] * x) + rng.normal(scale=0.025, size=x.size)

def residual(theta):
    return theta[0] * np.exp(theta[1] * x) - y

def jacobian(theta):
    e = np.exp(theta[1] * x)
    return np.column_stack([e, theta[0] * x * e])

def objective(theta):
    r = residual(theta)
    return 0.5 * float(r @ r)

def iterate(theta0, method, max_iter=12):
    theta = theta0.astype(float).copy()
    lamb = 1e-2
    rows = []
    for k in range(max_iter):
        r = residual(theta)
        J = jacobian(theta)
        H = J.T @ J
        g = J.T @ r
        if method == "lm":
            step = -np.linalg.solve(H + lamb * np.diag(np.diag(H)), g)
        else:
            step = -np.linalg.solve(H, g)
        trial = theta + step
        accepted = objective(trial) <= objective(theta)
        if method == "lm":
            if accepted:
                theta = trial
                lamb = max(lamb / 3.0, 1e-7)
            else:
                lamb = min(lamb * 8.0, 1e4)
        else:
            theta = trial
        rows.append({"iter": k, "a": float(theta[0]), "b": float(theta[1]), "objective": objective(theta), "lambda": float(lamb), "accepted": bool(accepted)})
    return rows

gn_rows = iterate(np.array([0.45, -0.12]), "gn")
lm_rows = iterate(np.array([0.45, -0.12]), "lm")

Agrid, Bgrid = np.meshgrid(np.linspace(0.35, 1.55, 120), np.linspace(-1.35, -0.05, 120))
Z = np.vectorize(lambda a, b: objective(np.array([a, b])))(Agrid, Bgrid)
fig, ax = plt.subplots(figsize=(7.2, 5.4))
cs = ax.contour(Agrid, Bgrid, Z, levels=18, cmap="viridis", alpha=0.75)
ax.clabel(cs, inline=True, fontsize=7, fmt="%.2g")
for rows, color, label in [(gn_rows, "#d28b26", "Gauss-Newton"), (lm_rows, "#2f6fbb", "Levenberg-Marquardt")]:
    pts = np.array([[r["a"], r["b"]] for r in rows])
    ax.plot(pts[:, 0], pts[:, 1], "o-", color=color, label=label)
ax.scatter([true_theta[0]], [true_theta[1]], marker="*", s=150, color="#b23a48", label="true parameters")
ax.set_xlabel("amplitude a")
ax.set_ylabel("decay b")
ax.set_title("Iterative estimation path on a nonlinear residual surface")
ax.legend()
path_plot = save_matplotlib(fig, TOPIC, "figures", "optimizer-path-comparison.png")
plt.close(fig)
artifact_paths.append(path_plot)
display_artifact(path_plot, width=780)


## 2. Jacobian and damping diagnostics

A local linear method is only as good as its Jacobian. The first check compares the analytic Jacobian with finite differences. The second plot shows the damping parameter and objective along the LM run, so the learner can inspect when the method trusts the local quadratic model and when it retreats.

In [ ]:
theta_check = np.array([1.05, -0.65])
J_analytic = jacobian(theta_check)
eps = 1e-6
J_fd = np.column_stack([(residual(theta_check + eps * np.eye(2)[i]) - residual(theta_check - eps * np.eye(2)[i])) / (2 * eps) for i in range(2)])
jacobian_error = float(np.max(np.abs(J_analytic - J_fd)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].semilogy([r["iter"] for r in lm_rows], [r["objective"] for r in lm_rows], "o-", color="#2f6fbb")
axes[0].set_title("accepted LM objectives")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("0.5 ||r||^2")
axes[0].grid(True, alpha=0.25)
axes[1].semilogy([r["iter"] for r in lm_rows], [r["lambda"] for r in lm_rows], "o-", color="#b23a48")
axes[1].set_title("damping adapts to the local model")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("lambda")
axes[1].grid(True, alpha=0.25)
fig.tight_layout()
damping_path = save_matplotlib(fig, TOPIC, "figures", "lm-damping-schedule.png")
plt.close(fig)
artifact_paths.append(damping_path)
display_artifact(damping_path, width=900)
print({"max_finite_difference_jacobian_error": jacobian_error})


## 3. Robust costs: outliers should lose leverage without disappearing

Robust estimation changes the local least-squares problem by reweighting residuals. Large residuals remain visible, but their weights shrink. The plot makes the weight rule inspectable and then applies it to one contaminated line fit, a small stand-in for image measurements with mismatches.

In [ ]:
res_grid = np.linspace(-4, 4, 300)
delta = 1.0
huber_weight = np.where(np.abs(res_grid) <= delta, 1.0, delta / np.abs(res_grid))

xr = np.linspace(-1.5, 1.5, 28)
Ar = np.column_stack([xr, np.ones_like(xr)])
yr = 0.8 * xr + 0.2 + rng.normal(scale=0.06, size=xr.size)
yr[[4, 20, 24]] += np.array([1.2, -1.0, 1.4])
theta = np.linalg.lstsq(Ar, yr, rcond=None)[0]
for _ in range(7):
    rr = Ar @ theta - yr
    scale = np.median(np.abs(rr)) / 0.6745 + 1e-9
    weights = np.where(np.abs(rr / scale) <= delta, 1.0, delta / np.abs(rr / scale))
    theta = np.linalg.lstsq(np.sqrt(weights)[:, None] * Ar, np.sqrt(weights) * yr, rcond=None)[0]
ordinary = np.linalg.lstsq(Ar, yr, rcond=None)[0]
robust_weights_min = float(weights.min())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
axes[0].plot(res_grid, huber_weight, color="#2f6fbb", lw=2.5)
axes[0].set_title("Huber weights shrink large residuals")
axes[0].set_xlabel("standardized residual")
axes[0].set_ylabel("weight")
axes[0].grid(True, alpha=0.25)
xx = np.linspace(xr.min(), xr.max(), 120)
axes[1].scatter(xr, yr, c=weights, cmap="viridis", s=45, edgecolor="#333")
axes[1].plot(xx, ordinary[0] * xx + ordinary[1], color="#d28b26", label="ordinary LS")
axes[1].plot(xx, theta[0] * xx + theta[1], color="#b23a48", label="robust IRLS")
axes[1].set_title("robust fit keeps outliers visible but lighter")
axes[1].legend()
axes[1].grid(True, alpha=0.25)
fig.tight_layout()
robust_path = save_matplotlib(fig, TOPIC, "figures", "robust-loss-weights.png")
plt.close(fig)
artifact_paths.append(robust_path)
display_artifact(robust_path, width=900)


## 4. Sparse normal equations and the Schur complement

Bundle adjustment is large because cameras and points are optimized together, but the residual graph is sparse: an observation touches one camera block and one point block. Eliminating point blocks gives a Schur complement on cameras. The numerical check below solves a small block system both densely and by Schur elimination and confirms that the camera step agrees.

In [ ]:
camera_blocks = 3
point_blocks = 5
cam_dim = 2
pt_dim = 2
obs = [(c, p) for c in range(camera_blocks) for p in range(point_blocks) if (c + p) % 2 == 0 or c == 0]
rows = []
cols = []
data = []
row = 0
for c, p in obs:
    for local in range(2):
        for j in range(cam_dim):
            rows.append(row + local); cols.append(c * cam_dim + j); data.append(0.8 + 0.1 * (j + local + 1))
        point_offset = camera_blocks * cam_dim + p * pt_dim
        for j in range(pt_dim):
            rows.append(row + local); cols.append(point_offset + j); data.append(0.5 + 0.07 * (j + local + 1))
    row += 2
J = sparse.coo_matrix((data, (rows, cols)), shape=(2 * len(obs), camera_blocks * cam_dim + point_blocks * pt_dim)).tocsr()
N = (J.T @ J + sparse.eye(J.shape[1]) * 0.2).toarray()
g = rng.normal(size=J.shape[1])
dense_step = np.linalg.solve(N, -g)
cc = camera_blocks * cam_dim
B = N[:cc, :cc]
E = N[:cc, cc:]
C = N[cc:, cc:]
gc = g[:cc]
gp = g[cc:]
S = B - E @ np.linalg.solve(C, E.T)
rhs = -gc + E @ np.linalg.solve(C, gp)
schur_camera_step = np.linalg.solve(S, rhs)
schur_dense_gap = float(np.linalg.norm(dense_step[:cc] - schur_camera_step))

G = nx.Graph()
for c in range(camera_blocks): G.add_node(f"C{c+1}", kind="camera")
for p in range(point_blocks): G.add_node(f"X{p+1}", kind="point")
for c, p in obs: G.add_edge(f"C{c+1}", f"X{p+1}")
pos = {f"C{c+1}": (0, -c) for c in range(camera_blocks)} | {f"X{p+1}": (2, -p + 1) for p in range(point_blocks)}
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.7))
axes[0].spy(N, markersize=5, color="#2f6fbb")
axes[0].set_title("block sparse normal matrix")
nx.draw_networkx(G, pos=pos, ax=axes[1], node_color=["#8ecae6" if n.startswith("C") else "#ffb703" for n in G.nodes], edge_color="#777", node_size=800, font_size=9)
axes[1].set_title("observation graph behind the blocks")
axes[1].axis("off")
fig.tight_layout()
schur_path = save_matplotlib(fig, TOPIC, "figures", "schur-complement-sparsity.png")
plt.close(fig)
artifact_paths.append(schur_path)
display_artifact(schur_path, width=900)


## Worked Example Reading Guide

The examples should be read as the control loop behind nonlinear vision estimation. The path plot shows that an iteration is a geometric move through parameter space, not a mysterious optimizer invocation. Gauss-Newton trusts the local least-squares model almost completely; Levenberg-Marquardt changes the same linearized system by adding damping, so bad local models lead to smaller, more cautious moves. The damping plot is therefore not just an algorithm trace. It is a diagnostic for how much the current quadratic approximation is being trusted.

The Jacobian check is deliberately explicit because many MVG failures are caused by derivatives that are off by a sign, a scale convention, or a homogeneous normalization. A finite-difference comparison is small and slow, but it is excellent as a notebook sanity check before a solver is trusted. The robust-weight panel makes a second diagnostic habit visible: outliers should lose influence without being erased from the plot. If the weights hide the bad measurements, the learner cannot tell whether the robust model is working or merely concealing a mismatch.

Finally, the Schur-complement view ties the appendix to camera/structure optimization. Eliminating point blocks is not just linear algebra cleverness; it follows the observation graph. Each image measurement touches a small set of variables, and the sparse normal matrix records that locality. The acceptance target is to inspect an optimizer run through its path, objective, damping, derivative check, robust weights, and block graph before trusting its final parameter vector.

## Applied Lab

Use the nonlinear residual pattern on a small geometry problem: fit a radial distortion coefficient, refine a homography by geometric reprojection error, or optimize a camera pose from 3D-2D correspondences. Keep the same audit trail: path plot, Jacobian check, damping/objective history, robust weights if outliers are present, and a sparse pattern if the variables split into blocks.

In [ ]:
lm_objectives = [row["objective"] for row in lm_rows]
accepted_descent = all(next_value <= current + 1e-12 for current, next_value in zip(lm_objectives, lm_objectives[1:]))
invariants = {
    "topic": TOPIC,
    "source_span": {"printed": "597-627", "pdf": "615-645"},
    "library_route": ["numpy.linalg", "scipy.sparse", "networkx", "matplotlib"],
    "artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in artifact_paths],
    "max_finite_difference_jacobian_error": jacobian_error,
    "lm_final_objective": float(lm_objectives[-1]),
    "lm_objective_monotone_after_acceptance": accepted_descent,
    "robust_min_weight": robust_weights_min,
    "schur_dense_camera_step_gap": schur_dense_gap,
    "normal_matrix_shape": list(N.shape),
    "normal_matrix_nnz": int(np.count_nonzero(N)),
}
summary_path = save_json(invariants, TOPIC, "checks", "iterative-estimation-invariants.json")
artifact_paths.append(summary_path)
display_artifact(summary_path)
assert_artifacts(artifact_paths, min_bytes=64)
assert invariants["max_finite_difference_jacobian_error"] < 1e-6
assert invariants["lm_final_objective"] < objective(np.array([0.45, -0.12]))
assert invariants["robust_min_weight"] < 0.6
assert invariants["schur_dense_camera_step_gap"] < 1e-10
final_sanity = invariants
invariants


## How To Use These Checks In A Chapter Notebook

When a chapter uses nonlinear refinement, this appendix gives the small audit trail that should accompany the final answer. A residual definition should state the measurement units, because a solver can only minimize the residual it is given. A Jacobian check should be run before interpreting convergence, because a wrong derivative can still produce a smooth-looking objective trace. A damping or trust-region trace should be kept when the starting estimate is rough, since the first few iterations reveal whether the local model is reliable. Robust weights should be plotted against residuals so outliers remain visible. For multi-camera problems, the sparsity graph should be shown before the solve: it tells the learner which variables are coupled by observations and why Schur elimination preserves the same solution while changing the computational route.

## Inspection Questions

Before accepting an iterative result, ask what changed at each step. Did the objective decrease for the intended residual, not just for a surrogate? Did the damping parameter respond when the local model became unreliable? Did robust weights reduce the leverage of large residuals while leaving those residuals visible? Did the sparse graph match the measurement design? These questions turn convergence from a black-box status message into an inspectable geometric argument.

The final habit is to keep the initial estimate in view. Nonlinear MVG problems often fail because the starting point lies outside the basin where the linearization is meaningful. A good notebook therefore shows both the starting residual and the final residual, so improvement is visible as a trajectory rather than asserted only by a final number.

## Pitfalls And Misconceptions

- A decreasing objective does not prove the geometric model is correct; it only proves the chosen residual got smaller.
- Gauss-Newton can be fast near the solution and reckless far from it.
- Damping is not a magic constant; it is a trust signal for the local quadratic model.
- Robust weights should reduce leverage without hiding outliers from the diagnostic plot.
- Sparse block structure belongs to the problem geometry, so flattening it too early makes the algorithm harder to reason about.

## Takeaways

- Iterative estimation is repeated local linearization plus disciplined step acceptance.
- Jacobian checks are cheap insurance against confident but wrong updates.
- Levenberg-Marquardt interpolates between cautious descent and fast Gauss-Newton behavior.
- Robust costs change influence, not the existence of the measurement.
- Sparse Schur structure is the computational reason large MVG optimizations are tractable.